In [1]:

import mlflow
import pandas as pd
import mlflow.sklearn
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pandas as pd
import re
import string
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import numpy as np

c:\Users\Shagufta\anaconda3\envs\atlas\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
df = pd.read_csv('IMDB.csv')
df = df.sample(500)
df.to_csv('data.csv', index=False)
df.head()

,review,sentiment
271,I want to state first that I am a Christian (a...,negative
388,Ostensibly a story about the young child of Ji...,positive
497,"Though derivative, ""Labyrinth"" still stands as...",positive
57,This film is worthwhile despite what you may h...,positive
946,1st watched 5/17/2002 - 3 out of 10(Dir-Ewald ...,negative


Data preprocessing

In [3]:
def lemmatization(text):
    """Lemmatize the text"""
    lemmatizer = WordNetLemmatizer()
    text = text.split()
    text = [lemmatizer.lemmatize(word) for word in text]
    return " ".join(text)



In [4]:
def remove_stop_words(text):
    """Remove stop words from the text."""
    stop_words = set(stopwords.words("english"))
    text = [word for word in str(text).split() if word not in stop_words]
    return " ".join(text)

In [5]:
def removing_numbers(text):
    """Remove numbers from the text."""
    text = ''.join([char for char in text if not char.isdigit()])
    return text

In [6]:
def lower_case(text):
    """Convert text to lower case."""
    text = text.split()
    text = [word.lower() for word in text]
    return " ".join(text)

In [7]:
def removing_punctuations(text):
    """Remove punctuations from the text."""
    text = re.sub('[%s]' % re.escape(string.punctuation), ' ', text)
    text = text.replace('؛', "")
    text = re.sub('\s+', ' ', text).strip()
    return text

In [8]:
def removing_urls(text):
    """Remove URLs from the text."""
    url_pattern = re.compile(r'https?://\S+|www\.\S+')
    return url_pattern.sub(r'', text)


In [9]:
def normalize_text(df):
    """Normalize the text data."""
    try:
        df['review'] = df['review'].apply(lower_case)
        df['review'] = df['review'].apply(remove_stop_words)
        df['review'] = df['review'].apply(removing_numbers)
        df['review'] = df['review'].apply(removing_punctuations)
        df['review'] = df['review'].apply(removing_urls)
        df['review'] = df['review'].apply(lemmatization)
        return df
    except Exception as e:
        print(f'Error during text normalization: {e}')
        raise

In [10]:
df = normalize_text(df)
df.head()

,review,sentiment
271,want state first christian and work film tv in...,negative
388,ostensibly story young child jimmy stewart dor...,positive
497,though derivative labyrinth still stand highli...,positive
57,film worthwhile despite may hear performance m...,positive
946,st watched dir ewald andre dupont fairly lame ...,negative


In [11]:
df['sentiment'].value_counts()

sentiment
negative    268
positive    232
Name: count, dtype: int64

In [12]:
x = df['sentiment'].isin(['positive','negative'])
df = df[x]

In [13]:
df.head()

,review,sentiment
271,want state first christian and work film tv in...,negative
388,ostensibly story young child jimmy stewart dor...,positive
497,though derivative labyrinth still stand highli...,positive
57,film worthwhile despite may hear performance m...,positive
946,st watched dir ewald andre dupont fairly lame ...,negative


In [14]:
df['sentiment'] = df['sentiment'].map({'positive':1, 'negative':0})
df.head()

,review,sentiment
271,want state first christian and work film tv in...,0
388,ostensibly story young child jimmy stewart dor...,1
497,though derivative labyrinth still stand highli...,1
57,film worthwhile despite may hear performance m...,1
946,st watched dir ewald andre dupont fairly lame ...,0


In [15]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [20]:
vectorizer = CountVectorizer(max_features=50)
X = vectorizer.fit_transform(df['review'])
y = df['sentiment']

In [21]:

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

In [22]:
import dagshub

import mlflow

mlflow.set_tracking_uri("https://dagshub.com/shagufta17/Movie-Review-Sentiment-Analysis.mlflow")
dagshub.init(repo_owner='shagufta17', repo_name='Movie-Review-Sentiment-Analysis', mlflow=True)

mlflow.set_experiment('Logistic Regression Baseline')

2026-05-13 10:35:51,410 - INFO - HTTP Request: GET https://dagshub.com/api/v1/repos/shagufta17/Movie-Review-Sentiment-Analysis "HTTP/1.1 200 OK"


Initialized MLflow to track repo "shagufta17/Movie-Review-Sentiment-Analysis"

2026-05-13 10:35:51,416 - INFO - Initialized MLflow to track repo "shagufta17/Movie-Review-Sentiment-Analysis"


Repository shagufta17/Movie-Review-Sentiment-Analysis initialized!

2026-05-13 10:35:51,418 - INFO - Repository shagufta17/Movie-Review-Sentiment-Analysis initialized!


<Experiment: artifact_location='mlflow-artifacts:/ac8c3bee7984453880042a26a7a9058f', creation_time=1778648257089, experiment_id='0', last_update_time=1778648257089, lifecycle_stage='active', name='Logistic Regression Baseline', tags={}, trace_location=None, workspace='default'>

In [23]:
import mlflow
import logging
import os
import time
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score


# Configure logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")

logging.info("Starting MLflow run...")

with mlflow.start_run():
    start_time = time.time()

    try:
        logging.info("Logging preprocessing parameters...")
        mlflow.log_param("vectorizer", "Bag of Words")
        mlflow.log_param("num_features", 100)
        mlflow.log_param("test_size", 0.25)

        logging.info("Initializing Logistic Regression model...")
        model = LogisticRegression(max_iter=1000)  # Increase max_iter to prevent non-convergence issues

        logging.info("Fitting the model...")
        model.fit(X_train, y_train)
        logging.info("Model training complete.")

        logging.info("Logging model parameters...")
        mlflow.log_param("model", "Logistic Regression")

        logging.info("Making predictions...")
        y_pred = model.predict(X_test)

        logging.info("Calculating evaluation metrics...")
        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred)
        recall = recall_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)

        logging.info("Logging evaluation metrics...")
        mlflow.log_metric("accuracy", accuracy)
        mlflow.log_metric("precision", precision)
        mlflow.log_metric("recall", recall)
        mlflow.log_metric("f1_score", f1)

        logging.info("Saving and logging the model...")
        mlflow.sklearn.log_model(model, "model")

        # Log execution time
        end_time = time.time()
        logging.info(f"Model training and logging completed in {end_time - start_time:.2f} seconds.")


        # Print the results for verification
        logging.info(f"Accuracy: {accuracy}")
        logging.info(f"Precision: {precision}")
        logging.info(f"Recall: {recall}")
        logging.info(f"F1 Score: {f1}")

    except Exception as e:
        logging.error(f"An error occurred: {e}", exc_info=True)

2026-05-13 10:36:07,342 - INFO - Model training and logging completed in 14.52 seconds.
2026-05-13 10:36:07,343 - INFO - Accuracy: 0.664
2026-05-13 10:36:07,343 - INFO - Precision: 0.6666666666666666
2026-05-13 10:36:07,344 - INFO - Recall: 0.576271186440678
2026-05-13 10:36:07,344 - INFO - F1 Score: 0.6181818181818182


🏃 View run painted-bat-671 at: https://dagshub.com/shagufta17/Movie-Review-Sentiment-Analysis.mlflow/#/experiments/0/runs/98a88892d66a4c49b23a32f45291c91f
🧪 View experiment at: https://dagshub.com/shagufta17/Movie-Review-Sentiment-Analysis.mlflow/#/experiments/0
